# Supplementary Figure 19 | chr3:35639573:G:A shifts ENSG00000295459 frontal cortex splicing from a cryptic to an annotated splice acceptor

**a.** Transcript models (junctions J1 = annotated intron chr3:35638953:35639754:clu_37423_-; J3 = cryptic intron chr3:35639634:35639754:clu_37423_-)

**b.** AlphaGenome (−) strand SPLICE_SITES relative change 100×(A−G)/G (positions kept only if the G score ≥ 0.01). The lead lies 61 bp from the J3 acceptor; AlphaGenome predicts a decrease at the cryptic acceptor for the A allele.

**c.** Frontal cortex RNA-seq coverage by genotype at CoS lead chr3:35639573:G:A (G/G, G/A, A/A).

**d.** STAR unique splice-junction counts for canonical acceptor J1 and cryptic acceptor J3 by genotype. The A allele increases J1 and decreases J3 usage.

**e.** Normalized LeafCutter intron-excision phenotypes for the same junctions by genotype.


## Setup


In [ ]:
import gzip
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import pyBigWig
import pysam
import seaborn as sns
from matplotlib.transforms import blended_transform_factory, offset_copy
from tqdm.auto import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.2, 5.0, 300
j1_color, j3_color = "#1B7F7A", "#C45C26"
tx_color = sns.color_palette("Paired", 12)[5]
arpp_color = sns.color_palette("Paired", 12)[1]
ncorf_color = "0.1"

cos_lead_variant = "chr3:35639573:G:A"
ag_ref_min = 0.01

# J1 annotated, J3 cryptic; last flag = primary (↑ with A)
sashimi_junctions = [
    ("J1", 35638954, 35639753, True),
    ("J3", 35639635, 35639753, False),
]
sqtl_phenotypes = [
    "chr3:35638953:35639754:clu_37423_-:ENSG00000295459.1",
    "chr3:35639634:35639754:clu_37423_-:ENSG00000295459.1",
]
junction_guide_xs = [35638953, 35639045, 35639634, 35639754]

focal_transcripts = [
    {
        "transcript_id": "ENST00000730226.1",
        "label": "ENST00000730226",
        "exons": [(35638662, 35638953), (35639754, 35640182)],
        "junctions": [
            {"coords": (35638953, 35639754), "label": "J1"},
            {"coords": (35639634, 35639754), "label": "J3"},
        ],
        "ncorf_exons": [(35638850, 35638953), (35639754, 35640154)],
    },
    {
        "transcript_id": "ENST00000730227.1",
        "label": "ENST00000730227",
        "exons": [(35638671, 35639045), (35639754, 35640178)],
        "junctions": [
            {"coords": (35639045, 35639754), "label": "J2"},
        ],
        "ncorf_exons": [(35638850, 35639045), (35639754, 35639825)],
    },
]

_ref, _alt = cos_lead_variant.split(":")[2:]
gt_order = [_ref + _ref, _ref + _alt, _alt + _alt]  # GG, GA, AA
sashimi_win = (35_638_500, 35_640_500)
ba9_smtsd = "Brain - Frontal Cortex (BA9)"

## Paths

In [ ]:
annot_feather = Path("path/to/gencode.v47.annotation.gtf.feather")
sashimi_leaf_pheno = Path(
    "path/to/Brain_Frontal_Cortex_BA9.v11.sQTL_phenotypes.bed.parquet"
)
# Protected / dbGaP inputs: WGS genotypes + per-sample RNA tracks.
sashimi_vcf = Path(
    "path/to/GTEx_Analysis_2021-02-11_v9_WholeGenomeSeq_953Indiv.SHAPEIT2_phased.vcf.gz"
)
gtex_sample_meta_file = Path("path/to/GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt")
sashimi_bw_dir = Path("path/to/gtex_coverage_bigwig")
sashimi_sj_dir = Path("path/to/gtex_star_sj")
ag_splice_sites = Path("path/to/chr3_35639573_G_A.splice_sites.locus.parquet")

fig_dir = Path("figures")


## Load data


### GENCODE annotation


In [ ]:
ann = pd.read_feather(annot_feather)
print(f"Annotation rows: {len(ann):,}")

### BA9 coverage and STAR junction counts by genotype

Built in-notebook from the WGS VCF + BA9 bigWigs / STAR `SJ.out.tab.gz` (protected).

In [ ]:
for required in (sashimi_bw_dir, sashimi_sj_dir, sashimi_vcf, gtex_sample_meta_file):
    if not Path(required).exists():
        raise FileNotFoundError(f"missing sashimi input: {required}")

meta = pd.read_csv(
    gtex_sample_meta_file,
    sep="\t",
    usecols=["SAMPID", "SMTSD", "SMAFRZE"],
    low_memory=False,
)
ba9 = meta[(meta["SMTSD"] == ba9_smtsd) & (meta["SMAFRZE"] == "RNASEQ")].copy()
ba9["subj"] = ba9["SAMPID"].astype(str).str.extract(r"^(GTEX-[^-]+)")[0]
rows = []
for _, g in ba9.groupby("subj"):
    ok = g[
        g["SAMPID"].map(
            lambda s: (sashimi_bw_dir / f"{s}.bigWig").exists()
            and (sashimi_sj_dir / f"{s}.SJ.out.tab.gz").exists()
        )
    ]
    if len(ok):
        rows.append(ok.iloc[0])
samples = pd.DataFrame(rows)
if samples.empty:
    raise RuntimeError("No BA9 RNA-seq samples with both bigWig and SJ.out.tab.gz")
print(f"BA9 RNA-seq samples with BW+SJ: {len(samples)}")

vf = pysam.VariantFile(str(sashimi_vcf))
vcf_samples = set(vf.header.samples)
chrom, pos, ref, alt = cos_lead_variant.split(":")
pos = int(pos)
recs = list(vf.fetch(chrom, pos - 1, pos))
rec = next(r for r in recs if r.pos == pos and r.ref == ref and alt in (r.alts or ()))
gser = {}
for subj in samples["subj"]:
    if subj not in vcf_samples:
        continue
    gt = rec.samples[subj].get("GT")
    if gt is None or None in gt:
        continue
    gser[subj] = gt_order[sum(1 for a in gt if a == 1)]
gser = pd.Series(gser, name="genotype")
samples = samples[samples["subj"].isin(gser.index)].copy()
samples["genotype"] = samples["subj"].map(gser)
print(samples["genotype"].value_counts().reindex(gt_order).to_string())

win_lo, win_hi = sashimi_win
width = win_hi - win_lo
cov = {g: np.zeros(width, dtype=float) for g in gt_order}
n = {g: 0 for g in gt_order}
junc = {g: {f"{a}-{b}": [] for _, a, b, _ in sashimi_junctions} for g in gt_order}
want_junc = {(int(a), int(b)) for _, a, b, _ in sashimi_junctions}

for row in tqdm(samples.itertuples(index=False), total=len(samples), desc="sashimi samples"):
    g = row.genotype
    bw = pyBigWig.open(str(sashimi_bw_dir / f"{row.SAMPID}.bigWig"))
    try:
        vals = bw.values("chr3", win_lo, win_hi, numpy=True)
    finally:
        bw.close()
    vals = np.asarray(vals, dtype=float)
    vals[np.isnan(vals)] = 0.0
    cov[g] += vals
    n[g] += 1

    counts = {ab: 0 for ab in want_junc}
    with gzip.open(sashimi_sj_dir / f"{row.SAMPID}.SJ.out.tab.gz", "rt") as fh:
        for line in fh:
            if not line.startswith("chr3\t"):
                continue
            p = line.split("\t")
            a, b = int(p[1]), int(p[2])
            if a > win_hi:
                break
            if (a, b) in counts:
                counts[(a, b)] = int(p[6])
    for _, a, b, _ in sashimi_junctions:
        junc[g][f"{a}-{b}"].append(counts.get((a, b), 0))

mean_cov = np.stack([cov[g] / n[g] if n[g] else cov[g] for g in gt_order])
junc_keys = [f"{a}-{b}" for _, a, b, _ in sashimi_junctions]
junc_samples = np.empty((len(gt_order), len(junc_keys)), dtype=object)
for gi, g in enumerate(gt_order):
    for ji, k in enumerate(junc_keys):
        junc_samples[gi, ji] = np.asarray(junc[g][k], dtype=float)

sashimi = {
    "win_lo": win_lo,
    "win_hi": win_hi,
    "genotypes": np.array(gt_order, dtype=object),
    "n_per_gt": np.array([n[g] for g in gt_order]),
    "mean_cov": mean_cov,
    "junc_keys": np.array(junc_keys, dtype=object),
    "junc_primary": np.array([p for *_, p in sashimi_junctions]),
    "junc_samples": junc_samples,
}
print(
    f"window chr3:{win_lo}-{win_hi}; n per genotype: "
    f"{dict(zip(list(sashimi['genotypes']), list(sashimi['n_per_gt'])))}"
)

### AlphaGenome splice sites


In [ ]:
ag = pd.read_parquet(ag_splice_sites)
ag = ag[
    (ag["pos"] >= win_lo)
    & (ag["pos"] <= win_hi)
    & (ag["track_strand"].astype(str) == "-")
].copy()
ag_by_allele = (
    ag.groupby(["pos", "allele"], as_index=False)["value"]
    .max()
    .pivot(index="pos", columns="allele", values="value")
    .sort_index()
)
for col in ("ref", "alt"):
    if col not in ag_by_allele.columns:
        ag_by_allele[col] = np.nan
ref_score = ag_by_allele["ref"].astype(float)
alt_score = ag_by_allele["alt"].astype(float)
ok = ref_score >= ag_ref_min
ag_rel_pct = pd.Series(np.nan, index=ag_by_allele.index, dtype=float)
ag_rel_pct.loc[ok] = 100.0 * (alt_score.loc[ok] - ref_score.loc[ok]) / ref_score.loc[ok]
ag_rel_pct = ag_rel_pct.dropna()
print(f"AG (−) splice-site positions with REF≥{ag_ref_min}: {len(ag_rel_pct):,}")

### LeafCutter phenotypes by genotype


In [ ]:
leaf = pd.read_parquet(sashimi_leaf_pheno, filters=[("ID", "in", list(sqtl_phenotypes))])
if leaf.empty:
    raise FileNotFoundError(f"No LeafCutter phenotypes in {sashimi_leaf_pheno}")
leaf = leaf.set_index("ID").reindex(sqtl_phenotypes)
leaf_sample_cols = [c for c in leaf.columns if str(c).startswith("GTEX-")]

vf = pysam.VariantFile(str(sashimi_vcf))
vcf_samples = set(vf.header.samples)
chrom, pos, ref, alt = cos_lead_variant.split(":")
pos = int(pos)
recs = list(vf.fetch(chrom, pos - 1, pos))
rec = next(r for r in recs if r.pos == pos and r.ref == ref and alt in (r.alts or ()))
leaf_gser = {}
for subj in leaf_sample_cols:
    if subj not in vcf_samples:
        continue
    gt = rec.samples[subj].get("GT")
    if gt is None or None in gt:
        continue
    leaf_gser[subj] = gt_order[sum(1 for a in gt if a == 1)]
leaf_gser = pd.Series(leaf_gser, name="genotype")

leaf_by_junc = []
for ph in sqtl_phenotypes:
    row = leaf.loc[ph, leaf_sample_cols].astype(float)
    by_gt = {g: [] for g in gt_order}
    for subj, val in row.items():
        if subj not in leaf_gser.index or not np.isfinite(val):
            continue
        by_gt[leaf_gser[subj]].append(float(val))
    leaf_by_junc.append({g: np.asarray(by_gt[g], dtype=float) for g in gt_order})
print("LeafCutter n by junction × genotype:")
for ji, (lab, *_) in enumerate(sashimi_junctions):
    print(f"  {lab}: " + ", ".join(f"{g}={len(leaf_by_junc[ji][g])}" for g in gt_order))

## Make figure


In [ ]:
genotypes = list(sashimi["genotypes"])
n_per = list(sashimi["n_per_gt"])
mean_cov = sashimi["mean_cov"]
junc_keys = list(sashimi["junc_keys"])
junc_primary = list(sashimi["junc_primary"])
junc_samples = sashimi["junc_samples"]
x = np.arange(win_lo, win_hi)
lead_pos = int(cos_lead_variant.split(":")[1])
n_gt = len(genotypes)

fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
outer = fig.add_gridspec(
    1, 2,
    width_ratios=[2.7, 1.2],
    wspace=0.28,
    left=0.16, right=0.98, top=0.96, bottom=0.08,
)
left = outer[0, 0].subgridspec(
    n_gt + 2, 1,
    height_ratios=[1.15, 0.80] + [0.75] * n_gt,
    hspace=0.12,
)
right = outer[0, 1].subgridspec(2, 1, hspace=0.35)
ax_tx = fig.add_subplot(left[0, 0])
ax_ag = fig.add_subplot(left[1, 0], sharex=ax_tx)
axes_gt = [fig.add_subplot(left[i + 2, 0], sharex=ax_tx) for i in range(n_gt)]
ax_box_star = fig.add_subplot(right[0, 0])
ax_box_leaf = fig.add_subplot(right[1, 0])

# --- a. transcript models ---
txs = list(reversed(focal_transcripts))
y_arpp = 0.90
y_txs = [0.50, 0.18][: len(txs)]
backbone_color, backbone_lw = "0.85", 0.4
min_exon_w = 0.006 * (win_hi - win_lo)
exon_h = 0.075
left_trans = blended_transform_factory(ax_tx.transAxes, ax_tx.transData)
label_x = -0.02

ex = ann[
    (ann["gene_name"] == "ARPP21")
    & (ann["Feature"] == "exon")
    & (ann["Chromosome"] == "chr3")
    & (ann["Start"] <= win_hi)
    & (ann["End"] >= win_lo)
]
ivs = sorted((int(a), int(b)) for a, b in zip(ex["Start"], ex["End"]))
arpp_exons = []
for a, b in ivs:
    if not arpp_exons or a > arpp_exons[-1][1] + 1:
        arpp_exons.append([a, b])
    else:
        arpp_exons[-1][1] = max(arpp_exons[-1][1], b)

ax_tx.text(
    label_x, y_arpp, "ARPP21 (+)", transform=left_trans,
    ha="right", va="center", fontsize=3.5, color=arpp_color, style="italic",
    clip_on=False, zorder=6,
)
ax_tx.plot([win_lo, win_hi], [y_arpp, y_arpp], color=backbone_color, lw=backbone_lw, solid_capstyle="butt", zorder=2)
for e0, e1 in arpp_exons:
    x0, x1 = max(e0, win_lo), min(e1, win_hi)
    if x1 <= x0:
        continue
    ax_tx.add_patch(plt.Rectangle(
        (x0, y_arpp - exon_h / 2), max(x1 - x0 + 1, min_exon_w), exon_h,
        facecolor=arpp_color, edgecolor="none", zorder=3, linewidth=0,
    ))

for tx, y in zip(txs, y_txs):
    ax_tx.text(
        label_x, y, f"{tx['label']} (−)", transform=left_trans,
        ha="right", va="center", fontsize=3.5, color=tx_color, style="italic",
        clip_on=False, zorder=6,
    )
    ax_tx.plot([win_lo, win_hi], [y, y], color=backbone_color, lw=backbone_lw, solid_capstyle="butt", zorder=2)
    for e0, e1 in sorted((int(a), int(b)) for a, b in tx["exons"]):
        x0, x1 = max(e0, win_lo), min(e1, win_hi)
        if x1 <= x0:
            continue
        ax_tx.add_patch(plt.Rectangle(
            (x0, y - exon_h / 2), max(x1 - x0 + 1, min_exon_w), exon_h,
            facecolor=tx_color, edgecolor="none", zorder=3, linewidth=0,
        ))
    for o0, o1 in tx["ncorf_exons"]:
        x0, x1 = max(int(o0), win_lo), min(int(o1), win_hi)
        if x1 <= x0:
            continue
        ax_tx.add_patch(plt.Rectangle(
            (x0, y - exon_h / 2), max(x1 - x0 + 1, min_exon_w * 0.5), exon_h,
            facecolor=ncorf_color, edgecolor="none", zorder=5, linewidth=0,
        ))
    for junc in tx["junctions"]:
        j0, j1 = int(junc["coords"][0]), int(junc["coords"][1])
        x0, x1 = max(j0, win_lo), min(j1, win_hi)
        if x1 <= x0:
            continue
        mid = 0.5 * (x0 + x1)
        peak_y = y + (0.12 if junc["label"] == "J3" else 0.20)
        if junc["label"] == "J1":
            jcolor = j1_color
        elif junc["label"] == "J3":
            jcolor = j3_color
        else:
            jcolor = tx_color
        ax_tx.plot([x0, mid, x1], [y, peak_y, y], color=jcolor, lw=0.55, solid_capstyle="round", zorder=6)
        ax_tx.text(
            mid, peak_y + 0.015, junc["label"],
            ha="center", va="bottom", fontsize=3.2, color="0.1",
            clip_on=False, zorder=7,
        )

ax_tx.set_xlim(win_lo, win_hi)
ax_tx.set_ylim(0.0, 1.12)
ax_tx.set_yticks([])
ax_tx.set_ylabel("")
ax_tx.tick_params(axis="x", bottom=False, labelbottom=False, length=0)
sns.despine(ax=ax_tx, left=True, bottom=True)

# --- b. AlphaGenome relative % change ---
c_up, c_dn = "#3B6EA5", "#9B3D8A"
if len(ag_rel_pct):
    pos = ag_rel_pct.index.to_numpy()
    y = ag_rel_pct.to_numpy(dtype=float)
    colors = np.where(y >= 0, c_up, c_dn)
    ax_ag.axhline(0, color="0.75", lw=0.4, zorder=1)
    ax_ag.vlines(pos, 0.0, y, colors=colors, lw=0.7, zorder=2)
    ax_ag.scatter(pos, y, s=16, c=colors, marker="o", zorder=3, linewidths=0)
    lim = max(float(np.nanmax(np.abs(y))) * 1.25, 1.0)
    ax_ag.set_ylim(-lim, lim)
else:
    ax_ag.set_ylim(-1, 1)

if win_lo <= lead_pos <= win_hi:
    tip_trans = offset_copy(ax_ag.transData, fig=fig, y=3.2, units="points")
    ax_ag.scatter(
        [lead_pos], [0.0], marker="v", s=22, color="0.15",
        zorder=7, clip_on=False, linewidths=0, transform=tip_trans,
    )
    ax_ag.text(
        lead_pos, 0.0, f"  {_ref}>{_alt}",
        ha="left", va="bottom", fontsize=4, color="0.15",
        clip_on=False, zorder=7,
    )
ax_ag.set_xlim(win_lo, win_hi)
ax_ag.set_ylabel(
    "AG splice sites(−)\n"
    r"% $\Delta = 100\times(A-G)/G$",
    rotation=90, fontsize=5, labelpad=3,
)
ax_ag.yaxis.set_major_locator(plt.MaxNLocator(nbins=4))
ax_ag.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}"))
sns.despine(ax=ax_ag, bottom=True)
ax_ag.tick_params(axis="x", labelbottom=False, length=0)

# Dotted guides at junction splice sites
for ax in [ax_tx, ax_ag, *axes_gt]:
    for xg in junction_guide_xs:
        if win_lo <= xg <= win_hi:
            ax.axvline(xg, color="0.55", ls=":", lw=0.7, zorder=2)

# --- c. coverage by genotype ---
ymax = max(float(np.nanmax(mean_cov)) * 1.25, 1.0)
for ax, g, n_g, cov in zip(axes_gt, genotypes, n_per, mean_cov):
    ax.fill_between(x, 0, cov, color="0.75", linewidth=0, alpha=0.85, step="mid")
    ax.set_ylim(0, ymax)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=3))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}"))
    ax.set_ylabel(
        f"Coverage\n({g}, n={int(n_g)})",
        rotation=90, fontsize=5, labelpad=2,
    )
    is_last = ax is axes_gt[-1]
    sns.despine(ax=ax, bottom=not is_last)
    ax.tick_params(axis="x", labelbottom=is_last, length=3 if is_last else 0)

axes_gt[-1].set_xlim(win_lo, win_hi)
axes_gt[-1].set_xlabel("Genomic position", fontsize=6, labelpad=2)
axes_gt[-1].xaxis.set_major_locator(plt.MaxNLocator(nbins=4))
axes_gt[-1].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))

# --- d/e. STAR SJ + LeafCutter violins ---
star_by_junc = [
    {g: np.asarray(junc_samples[gi, ji], dtype=float) for gi, g in enumerate(genotypes)}
    for ji in range(len(junc_keys))
]
junc_order = sorted(range(len(junc_primary)), key=lambda i: (0 if junc_primary[i] else 1))
pair_gap, group_gap = 0.95, 2.7

for ax, series_by_junc, ylabel in [
    (ax_box_star, star_by_junc, "STAR SJ count"),
    (ax_box_leaf, leaf_by_junc, "LeafCutter phenotype"),
]:
    positions, series, colors = [], [], []
    xtick_pos, xtick_lab = [], []
    for gi, g in enumerate(genotypes):
        base = gi * group_gap
        n_g = None
        for k, ji in enumerate(junc_order):
            c = j1_color if junc_primary[ji] else j3_color
            vals = np.asarray(series_by_junc[ji][g], dtype=float)
            vals = vals[np.isfinite(vals)]
            positions.append(base + k * pair_gap)
            series.append(vals)
            colors.append(c)
            if n_g is None:
                n_g = int(len(vals))
        xtick_pos.append(base + 0.5 * pair_gap * (len(junc_order) - 1))
        g = str(g)
        xtick_lab.append(f"{g[0]}/{g[1]}\n({n_g})" if len(g) == 2 else f"{g}\n({n_g})")

    parts = ax.violinplot(
        series, positions=positions, widths=0.85,
        showmeans=False, showmedians=False, showextrema=False,
    )
    for body, c in zip(parts["bodies"], colors):
        body.set_facecolor(c)
        body.set_edgecolor("0.1")
        body.set_linewidth(0.5)
        body.set_alpha(0.85)
    for xp, vals in zip(positions, series):
        if len(vals) == 0:
            continue
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        ax.plot([xp, xp], [q1, q3], color="0.1", lw=1.0, solid_capstyle="butt", zorder=5)
        ax.scatter([xp], [med], s=1.2, color="white", edgecolors="0.1", linewidths=0.2, zorder=6)

    ax.set_xticks(xtick_pos)
    ax.set_xticklabels(xtick_lab, fontsize=4.0)
    ax.set_xlim(positions[0] - 0.9, positions[-1] + 0.9)
    ax.set_ylabel(ylabel, fontsize=4)
    ax.tick_params(axis="y", labelsize=4, length=1.8, width=0.45, pad=1)
    ax.tick_params(axis="x", labelsize=4, length=1.8, width=0.45, pad=1)
    ax.legend(
        handles=[
            mpatches.Patch(facecolor=j1_color, edgecolor="0.1", linewidth=0.5, label="J1"),
            mpatches.Patch(facecolor=j3_color, edgecolor="0.1", linewidth=0.5, label="J3"),
        ],
        loc="lower center", ncol=2,
        frameon=False, fontsize=4.0, handlelength=0.9,
        handletextpad=0.3, columnspacing=0.9, borderaxespad=0.2,
        bbox_to_anchor=(0.5, 1.0),
    )
    sns.despine(ax=ax)
    ax.spines["left"].set_linewidth(0.7)
    ax.spines["bottom"].set_linewidth(0.7)

# Panel letters
for ax, lab in ((ax_tx, "a"), (ax_ag, "b"), (axes_gt[0], "c")):
    ax.text(
        -.2, 1.05, lab, transform=ax.transAxes,
        fontsize=9, va="bottom", ha="right", clip_on=False, fontweight="bold",
    )
ax_box_star.text(
    -0.22, 1.05, "d", transform=ax_box_star.transAxes,
    fontsize=9, va="bottom", ha="right", clip_on=False, fontweight="bold",
)
ax_box_leaf.text(
    -0.22, 1.05, "e", transform=ax_box_leaf.transAxes,
    fontsize=9, va="bottom", ha="right", clip_on=False, fontweight="bold",
)

plt.show()

## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "supp_figure_19_ensg00000295459_sashimi.pdf", bbox_inches="tight")
fig.savefig(fig_dir / "supp_figure_19_ensg00000295459_sashimi.png", bbox_inches="tight")
print(f"Wrote {fig_dir / 'supp_figure_19_ensg00000295459_sashimi.pdf'}")
print(f"Wrote {fig_dir / 'supp_figure_19_ensg00000295459_sashimi.png'}")

## Print stats


In [ ]:
# Caption: lead distance to cryptic J3 acceptor (also printed in Fig. 5)
lead_pos = int(cos_lead_variant.split(":")[1])
j3_acceptor = 35_639_634  # cryptic intron start / acceptor on − strand annotation coords
print(f"Lead {cos_lead_variant}: {lead_pos - j3_acceptor} bp from J3 acceptor")
